In [ ]:
from google.colab import drive
drive.mount("/content/drive")
ROOT = "/content/drive/MyDrive/cv-longtail"

import sys, json
sys.path.append(ROOT)
import numpy as np
import torch
import torch.nn.functional as F
from torchvision.datasets import CIFAR10

train = CIFAR10(root=f"{ROOT}/data", train=True, download=True)
split = json.load(open(f"{ROOT}/splits/c10lt_rho100_seed0.json"))
lt_idx, counts = split["indices"], split["counts"]

Mounted at /content/drive


**Class-Balanced Validation Set**

In [ ]:
def make_val_split(lt_indices, targets, C=10, n_val=10, seed=0):
    rng = np.random.RandomState(seed)
    targets = np.array(targets)
    idx = np.array(lt_indices)
    val = []
    for c in range(C):
        cls = idx[targets[idx] == c]
        val += rng.choice(cls, n_val, replace=False).tolist()
    val = sorted(val)                         # val is validation indecies (balanced)
    tr = sorted(set(lt_indices) - set(val))   # tr is train indecies (long-tailed)
    return tr, val

tr_idx, val_idx = make_val_split(lt_idx, train.targets)

# checks
print(len(tr_idx), len(val_idx))                     # expect 12306 100
print(len(set(tr_idx) & set(val_idx)))               # expect 0 (no overlap)
print(np.bincount(np.array(train.targets)[val_idx])) # expect ten 10s
print(np.bincount(np.array(train.targets)[tr_idx]))  # counts minus 10 each

train_counts = np.bincount(np.array(train.targets)[tr_idx]).tolist()
with open(f"{ROOT}/splits/c10lt_rho100_seed0_val.json", "w") as f:
    json.dump({"train_indices": tr_idx, "val_indices": val_idx,
               "train_counts": train_counts}, f)

12306 100
0
[10 10 10 10 10 10 10 10 10 10]
[4990 2987 1786 1067  635  377  222  129   73   40]


**Counts and Groups**

In [ ]:
def make_groups(class_counts, many_thr=100, few_thr=20):  # standard thresholds from the papers
    many   = [c for c, n in enumerate(class_counts) if n > many_thr]
    few    = [c for c, n in enumerate(class_counts) if n < few_thr]
    medium = [c for c, n in enumerate(class_counts) if few_thr <= n <= many_thr]
    return {"many": many, "medium": medium, "few": few}
# make_groups works for CIFAR-10-LT but gives an empty "few" group
# instead use groups_c10
# a 10-val-split would leave the few (5) class empty
# instead tune on CIFAR-10-LT

C100 = 100
counts_c100 = [int(500 * 100 ** (-i / (C100 - 1))) for i in range(C100)]
groups_c100 = make_groups(counts_c100)

print(sum(counts_c100), counts_c100[0], counts_c100[-1])
print({k: len(v) for k, v in groups_c100.items()})
print(groups_c100["many"][-1], groups_c100["medium"][0], groups_c100["medium"][-1], groups_c100["few"][0])

10847 500 5
{'many': 35, 'medium': 35, 'few': 30}
34 35 69 70


In [ ]:
%%writefile -a /content/drive/MyDrive/cv-longtail/utils.py


def make_groups(class_counts, many_thr=100, few_thr=20):
    many   = [c for c, n in enumerate(class_counts) if n > many_thr]
    few    = [c for c, n in enumerate(class_counts) if n < few_thr]
    medium = [c for c, n in enumerate(class_counts) if few_thr <= n <= many_thr]
    return {"many": many, "medium": medium, "few": few}


counts_c100 = [int(500 * 100 ** (-i / 99)) for i in range(100)]
groups_c100 = make_groups(counts_c100)

Appending to /content/drive/MyDrive/cv-longtail/utils.py


**logit adjustment and balanced softmax (tau = 1)**

In [6]:
import torch.nn.functional as F

In [7]:
def log_prior(class_counts):
    counts = torch.tensor(class_counts, dtype=torch.float)
    return torch.log(counts / counts.sum())

def logit_adjusted_loss(logits, targets, lp, tau=1.0):
    return F.cross_entropy(logits + tau * lp.to(logits.device), targets)

def posthoc_adjust(logits, class_counts, tau=1.0):
    return logits - tau * log_prior(class_counts).to(logits.device)

In [8]:
counts = [5000, 2997, 1796, 1077, 645, 387, 232, 139, 83, 50]
lp = log_prior(counts)
logits = torch.randn(8, 10)
targets = torch.randint(0, 10, (8,))

# 1. tau = 0 gives plain CE
print(torch.allclose(logit_adjusted_loss(logits, targets, lp, tau=0.0),
                     F.cross_entropy(logits, targets)))

# 2. uniform prior gives plain CE (a constant shift does not change softmax)
lp_uniform = log_prior([100] * 10)
print(torch.allclose(logit_adjusted_loss(logits, targets, lp_uniform),
                     F.cross_entropy(logits, targets)))

# 3. priors sum to 1
print(lp.exp().sum())                      # expect 1.0

# 4. post-hoc: equal logits -> the rarest class (9) wins, since it has the smallest prior
print(posthoc_adjust(torch.zeros(1, 10), counts).argmax(dim=1))   # expect tensor([9])

# 5. post-hoc with uniform counts changes no predictions
print((posthoc_adjust(logits, [100] * 10).argmax(1) == logits.argmax(1)).all())

# 6. a head-class mistake costs less than the same mistake on a tail class
z = torch.zeros(2, 10)
print(logit_adjusted_loss(z[:1], torch.tensor([0]), lp).item(),
      logit_adjusted_loss(z[1:], torch.tensor([9]), lp).item())

True
True
tensor(1.)
tensor([9])
tensor(True)
0.9087421894073486 5.513912200927734


In [9]:
%%writefile -a /content/drive/MyDrive/cv-longtail/losses.py


def log_prior(class_counts):
    counts = torch.tensor(class_counts, dtype=torch.float)
    return torch.log(counts / counts.sum())


def logit_adjusted_loss(logits, targets, lp, tau=1.0):
    return F.cross_entropy(logits + tau * lp.to(logits.device), targets)


def posthoc_adjust(logits, class_counts, tau=1.0):
    return logits - tau * log_prior(class_counts).to(logits.device)


_get_loss_base = get_loss

def get_loss(name, class_counts, **kwargs):
    if name == "la":
        return partial(logit_adjusted_loss, lp=log_prior(class_counts),
                       tau=kwargs.get("tau", 1.0))
    if name == "balanced_softmax":
        return partial(logit_adjusted_loss, lp=log_prior(class_counts), tau=1.0)
    return _get_loss_base(name, class_counts, **kwargs)

Appending to /content/drive/MyDrive/cv-longtail/losses.py
